# 1. Cityscapes: полный запуск эксперимента в Google Colab

Ноутбук только настраивает окружение и вызывает скрипты проекта. Датасет, обучение, метрики и оценка остаются в `src/`. Cityscapes загружается через `kagglehub.dataset_download("electraawais/cityscape-dataset")`. Перед запуском выберите **Runtime → Change runtime type → GPU**.


## 1.1 Проверка GPU


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU не найден. Выберите Runtime → Change runtime type → GPU.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
!nvidia-smi

## 1.2 Подключение Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub

path = kagglehub.dataset_download("electraawais/cityscape-dataset")
print('KaggleHub dataset:', path)

In [ ]:
import copy
import os
from pathlib import Path

REPO_URL = "https://github.com/atikkhon/diploma.git"
CITYSCAPES_ROOT = Path(path).expanduser().resolve()

PROJECT_DIR = Path('/content/Diploma')
DRIVE_ROOT = Path('/content/drive/MyDrive/cityscapes_robustness')
RUNS_DIR = DRIVE_ROOT / 'runs'
MODELS_DIR = DRIVE_ROOT / 'models'
LOG_DIR = DRIVE_ROOT / 'logs'
TARGET_BRANCH = 'codex/no-mlflow-pipeline'
SPLIT_MANIFEST = DRIVE_ROOT / 'split_manifest.csv'
RUNTIME_CONFIG = DRIVE_ROOT / 'experiment_colab.yaml'

for directory in (DRIVE_ROOT, RUNS_DIR, MODELS_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    'REPO_URL': REPO_URL,
    'PROJECT_DIR': str(PROJECT_DIR),
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'RUNS_DIR': str(RUNS_DIR),
    'MODELS_DIR': str(MODELS_DIR),
    'LOG_DIR': str(LOG_DIR),
    'SPLIT_MANIFEST': str(SPLIT_MANIFEST),
    'RUNTIME_CONFIG': str(RUNTIME_CONFIG),
    'TARGET_BRANCH': TARGET_BRANCH,
})
print('Drive root:', DRIVE_ROOT)
print('Runs:', RUNS_DIR)
print('Models:', MODELS_DIR)
print('Git branch:', TARGET_BRANCH)


## 1.3 Клонирование или обновление Git-репозитория

При повторном запуске выполняется `git pull --ff-only`; локальные изменения в `/content/Diploma` не перезаписываются.


In [ ]:
%%bash
set -euo pipefail
if [[ -d "$PROJECT_DIR/.git" ]]; then
  git -C "$PROJECT_DIR" fetch origin "$TARGET_BRANCH" 2>&1 | tee -a "$LOG_DIR/git_update.log"
  git -C "$PROJECT_DIR" switch "$TARGET_BRANCH" 2>&1 | tee -a "$LOG_DIR/git_update.log"
  git -C "$PROJECT_DIR" pull --ff-only 2>&1 | tee -a "$LOG_DIR/git_update.log"
else
  git clone --branch "$TARGET_BRANCH" --single-branch "$REPO_URL" "$PROJECT_DIR" 2>&1 | tee -a "$LOG_DIR/git_update.log"
fi


## 1.4 Установка `requirements.txt`


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pip install -r requirements.txt 2>&1 | tee -a "$LOG_DIR/install.log"

## 1.5 Пути Cityscapes и Colab-конфигурация

Manifest, checkpoint, история обучения, оценки и логи переживают разрыв runtime, потому что сохраняются в Google Drive.


In [ ]:
import sys
import yaml

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.dataset import discover_cityscapes_layout, prepare_train_id_masks

layout = discover_cityscapes_layout(CITYSCAPES_ROOT)
# Маски читаются на каждой эпохе, поэтому держим кэш на локальном SSD Colab.
# После обрыва runtime эта ячейка быстро восстановит его из Kaggle labelIds.
prepared_gtfine = Path('/content/prepared_gtFine')
train_masks = prepare_train_id_masks(
    layout['train_masks'], prepared_gtfine / 'train'
)
val_masks = prepare_train_id_masks(
    layout['val_masks'], prepared_gtfine / 'val'
)

with (PROJECT_DIR / 'configs/experiment.yaml').open(encoding='utf-8') as file:
    config = yaml.safe_load(file)
config['data']['root'] = str(CITYSCAPES_ROOT)
config['data']['train_images'] = str(layout['train_images'])
config['data']['train_masks'] = str(train_masks)
config['data']['official_val_images'] = str(layout['val_images'])
config['data']['official_val_masks'] = str(val_masks)
config['data']['split_file'] = str(SPLIT_MANIFEST)
config['training']['log_interval'] = 25
with RUNTIME_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)
print('Runtime config:', RUNTIME_CONFIG)
print('Train images:', layout['train_images'])
print('Train masks:', train_masks)
print('Official val images:', layout['val_images'])
print('Official val masks:', val_masks)


## 1.6 Создание `split_manifest.csv`

Manifest пересоздаётся детерминированно, чтобы в нём всегда были актуальные локальные пути. При повторном запуске полная проверка масок не дублируется.


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -s "$SPLIT_MANIFEST" ]]; then
  echo "Обновление путей существующего manifest без повторной полной проверки масок" | tee -a "$LOG_DIR/create_split.log"
  python scripts/create_split.py --config "$RUNTIME_CONFIG" --skip-mask-validation 2>&1 | tee -a "$LOG_DIR/create_split.log"
else
  python scripts/create_split.py --config "$RUNTIME_CONFIG" 2>&1 | tee -a "$LOG_DIR/create_split.log"
fi

## 1.7 Запуск тестов


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -v 2>&1 | tee -a "$LOG_DIR/pytest.log"

## 1.8 Проверка датасета


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/smoke_test_dataset.py --config "$RUNTIME_CONFIG" --split dev 2>&1 | tee -a "$LOG_DIR/dataset_smoke_test.log"

In [ ]:
from IPython.display import Image, display
smoke_png = PROJECT_DIR / 'outputs/figures/dataset_smoke_test.png'
if not smoke_png.is_file():
    raise FileNotFoundError(f'Smoke-test PNG не найден: {smoke_png}')
display(Image(filename=str(smoke_png)))

## 1.9 Проверка ветки без MLflow

Ноутбук ещё раз обновляет выбранную ветку и запускает тесты проекта.


In [ ]:
%%bash
set -euo pipefail
git -C "$PROJECT_DIR" fetch origin codex/no-mlflow-pipeline
git -C "$PROJECT_DIR" switch codex/no-mlflow-pipeline
git -C "$PROJECT_DIR" pull --ff-only
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -q

## 1.10 Вспомогательная ячейка запуска модели

Запустите эту вспомогательную ячейку один раз после раздела 1.8. Каждая секция модели ниже вызывает её со своими настройками.


In [ ]:
import os
import subprocess
import yaml
import matplotlib.pyplot as plt
from IPython.display import display
from scripts.visualize_checkpoint import visualize_checkpoints
from src.visualization import create_training_curve_figures


CORRUPTION_CONFIG = {
    'darkness': {
        'levels': {
            1: {'factor': 0.75},
            2: {'factor': 0.55},
            3: {'factor': 0.35},
        }
    },
    'brightness': {
        'levels': {
            1: {'factor': 1.15},
            2: {'factor': 1.35},
            3: {'factor': 1.60},
        }
    },
    'gaussian_blur': {
        'levels': {
            1: {'kernel_size': 5, 'sigma': 1.0},
            2: {'kernel_size': 9, 'sigma': 2.0},
            3: {'kernel_size': 13, 'sigma': 3.0},
        }
    },
    'gaussian_noise': {
        'levels': {
            1: {'sigma': 8.0},
            2: {'sigma': 16.0},
            3: {'sigma': 28.0},
        }
    },
    'jpeg_compression': {
        'levels': {
            1: {'quality': 70},
            2: {'quality': 40},
            3: {'quality': 15},
        }
    },
    'fog': {
        'levels': {
            1: {'alpha': 0.15},
            2: {'alpha': 0.30},
            3: {'alpha': 0.45},
        }
    },
}


BASELINE_AUGMENTATION = {
    'policy': 'baseline',
    'horizontal_flip_probability': 0.5,
    'robust_one_of_probability': 0.5,
    'darkness': {
        'enabled': True,
        'min_factor': 0.55,
        'max_factor': 0.95,
    },
    'brightness': {
        'enabled': True,
        'min_factor': 1.05,
        'max_factor': 1.60,
    },
    'gaussian_blur': {
        'enabled': True,
        'kernel_sizes': [3, 5],
        'sigma_min': 0.3,
        'sigma_max': 1.2,
    },
    'gaussian_noise': {
        'enabled': True,
        'sigma_min': 3.0,
        'sigma_max': 10.0,
    },
    'jpeg_compression': {
        'enabled': True,
        'quality_min': 40,
        'quality_max': 85,
    },
}

ROBUST_AUGMENTATION = copy.deepcopy(BASELINE_AUGMENTATION)
ROBUST_AUGMENTATION['policy'] = 'robust'

ROBUST_AUGMENTATION_INFO = {
    'seen_corruptions': [
        'darkness',
        'brightness',
        'gaussian_blur',
        'gaussian_noise',
        'jpeg_compression',
    ],
    'unseen_corruptions': ['fog'],
}


def prepare_model_run(
    selected_model,
    run_name,
    resume_training,
    continue_from_run,
    init_checkpoint,
    model_settings,
    update_corruption_settings=False,
):
    if resume_training and continue_from_run is not None:
        raise ValueError('Choose only one mode: resume_training or continue_from_run')
    if init_checkpoint not in {'last', 'best'}:
        raise ValueError("init_checkpoint must be 'last' or 'best'")

    run_dir = DRIVE_ROOT / 'runs' / run_name
    model_dir = DRIVE_ROOT / 'models' / run_name
    run_config_path = run_dir / 'run_config.yaml'
    run_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    if resume_training and run_config_path.exists():
        with run_config_path.open(encoding='utf-8') as file:
            run_config = yaml.safe_load(file)

        config_updated = False
        if run_config.pop('robust_training', None) is not None:
            config_updated = True
            print('Removed legacy robust_training settings')
        if 'save_predictions' in run_config.get('evaluation', {}):
            run_config['evaluation'].pop('save_predictions')
            config_updated = True
            print('Removed unused evaluation.save_predictions setting')
        if update_corruption_settings:
            run_config['corruptions'] = CORRUPTION_CONFIG
            config_updated = True
            print('Replaced corruption settings from CORRUPTION_CONFIG')
        else:
            if 'corruptions' not in run_config:
                run_config['corruptions'] = {}
            added_corruptions = []
            for corruption_name, corruption_settings in CORRUPTION_CONFIG.items():
                if corruption_name not in run_config['corruptions']:
                    run_config['corruptions'][corruption_name] = corruption_settings
                    added_corruptions.append(corruption_name)
            if added_corruptions:
                config_updated = True
                print('Added missing corruption settings:', ', '.join(added_corruptions))

        if config_updated:
            with run_config_path.open('w', encoding='utf-8') as file:
                yaml.safe_dump(run_config, file, allow_unicode=True, sort_keys=False)
        print('Resume uses the existing run_config.yaml without overwriting training settings')
    else:
        with RUNTIME_CONFIG.open(encoding='utf-8') as file:
            run_config = yaml.safe_load(file)
        if run_config.pop('robust_training', None) is not None:
            print('Removed legacy robust_training settings from runtime config template')
        run_config['project_root'] = str(PROJECT_DIR)
        run_config['run'] = {
            'name': run_name,
            'output_dir': str(run_dir),
            'model_dir': str(model_dir),
        }
        run_config['run'].update(model_settings.get('run', {}))
        run_config['model'] = model_settings['model']
        run_config['training'].update(model_settings['training'])
        run_config['evaluation'].update(model_settings['evaluation'])
        run_config['evaluation'].pop('save_predictions', None)
        run_config['augmentation'] = model_settings.get('augmentation', BASELINE_AUGMENTATION)
        if run_config['augmentation'].get('policy') == 'robust':
            run_config['robust_augmentation'] = model_settings.get('robust_augmentation', ROBUST_AUGMENTATION_INFO)
        else:
            run_config.pop('robust_augmentation', None)
        for key in ('checkpoint_dir', 'history_dir'):
            run_config['training'].pop(key, None)
        for key in ('metrics_dir', 'figures_dir', 'predictions_dir'):
            run_config['evaluation'].pop(key, None)
        run_config['corruptions'] = CORRUPTION_CONFIG
        with run_config_path.open('w', encoding='utf-8') as file:
            yaml.safe_dump(run_config, file, allow_unicode=True, sort_keys=False)

    os.environ['RUN_CONFIG'] = str(run_config_path)
    os.environ['RUN_NAME'] = run_name
    os.environ['RUN_DIR'] = str(run_dir)
    os.environ['MODEL_DIR'] = str(model_dir)
    if 'IMAGE_INDEX' not in globals():
        globals()['IMAGE_INDEX'] = 0
    os.environ['IMAGE_INDEX'] = str(globals()['IMAGE_INDEX'])
    os.environ['RESUME_TRAINING'] = '1' if resume_training else '0'
    os.environ['CONTINUE_FROM_RUN'] = '' if continue_from_run is None else str(continue_from_run)
    os.environ['INIT_CHECKPOINT'] = init_checkpoint

    globals().update(
        {
            'SELECTED_MODEL': selected_model,
            'RUN_NAME': run_name,
            'RUN_DIR': run_dir,
            'MODEL_DIR': model_dir,
            'RUN_CONFIG': run_config_path,
            'RESUME_TRAINING': resume_training,
            'CONTINUE_FROM_RUN': continue_from_run,
            'INIT_CHECKPOINT': init_checkpoint,
            'UPDATE_CORRUPTION_SETTINGS': update_corruption_settings,
        }
    )

    print('Model:', selected_model)
    print('Run:', run_name)
    print('Results dir:', run_dir)
    print('Model dir:', model_dir)
    print('Config:', run_config_path)
    print('Resume:', resume_training)
    print('Continue from run:', continue_from_run)
    print('Init checkpoint:', init_checkpoint)
    print('Update corruption settings:', update_corruption_settings)
    print('Configured epochs:', run_config['training']['epochs'])


PREVIEW_CONDITIONS = {
    'clean',
    'darkness',
    'brightness',
    'gaussian_blur',
    'gaussian_noise',
    'jpeg_compression',
    'fog',
}


def run_command_to_log(command, log_path):
    print(' '.join(command))
    completed = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    print(completed.stdout)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed.stdout


def preview_segmentation(condition, image_indices, severity=None):
    if condition not in PREVIEW_CONDITIONS:
        raise ValueError(f'Unknown preview condition: {condition}')
    if not image_indices:
        raise ValueError('image_indices must contain at least one image index')
    if condition == 'clean' and severity is not None:
        raise ValueError('Clean preview does not use severity')
    if condition != 'clean' and severity not in {1, 2, 3}:
        raise ValueError('Corruption preview severity must be 1, 2, or 3')

    selected_indices = [int(image_index) for image_index in image_indices]
    print(
        f'Preview in Colab only: condition={condition}, '
        f'severity={severity}, indices={selected_indices}'
    )
    figures = visualize_checkpoints(
        RUN_CONFIG,
        selected_indices,
        condition=condition,
        severity=severity,
    )
    for figure in figures:
        display(figure)
        plt.close(figure)


## 1.11 U-Net

Сверните этот раздел, когда работаете с другой моделью.


### 1.11.1 U-Net: параметры


In [ ]:
UNET_SETTINGS = {
    'model': {
        'name': 'unet',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    'run': {
        'kind': 'baseline',
        'source_baseline_run': None,
    },
    'augmentation': BASELINE_AUGMENTATION,
    'training': {
        'epochs': 8,
        'batch_size': 8,
        'num_workers': 2,
        'log_interval': 25,
        'learning_rate': 0.0003,
        'weight_decay': 0.0001,
        'device': 'auto',
        'mixed_precision': True,
    },
    'evaluation': {
        'batch_size': 8,
    },
}

prepare_model_run(
    selected_model='unet',
    run_name='unet_experiment_01',
    resume_training=False,
    continue_from_run=None,
    init_checkpoint='last',
    model_settings=UNET_SETTINGS,
    update_corruption_settings=False,
)


### 1.11.2 U-Net: обучение, resume или continue


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -n "$CONTINUE_FROM_RUN" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --continue-from-run "$CONTINUE_FROM_RUN" --init-checkpoint "$INIT_CHECKPOINT" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
elif [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi


In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


### 1.11.3 U-Net: clean-оценка и предпросмотры


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"


In [ ]:
CLEAN_PREVIEW_INDICES = [0]
preview_segmentation('clean', CLEAN_PREVIEW_INDICES)


### 1.11.4 U-Net: сохранённые результаты


In [ ]:
import pandas as pd

print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
history_path = RUN_DIR / 'metrics' / 'training_history.csv'
display(pd.read_csv(history_path))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

for figure in create_training_curve_figures(history_path):
    display(figure)
    plt.close(figure)


### 1.11.5 U-Net: ручные оценки и предпросмотры искажений

В этих блоках вручную выбирается severity и список индексов для preview каждого искажения.


#### 1.11.5.1 U-Net: Darkness: оценка


In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY must be 1, 2, or 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


#### 1.11.5.2 U-Net: Darkness: предпросмотры сегментации


In [ ]:
DARKNESS_PREVIEW_INDICES = [0]
DARKNESS_SEVERITY = globals().get('DARKNESS_SEVERITY', 1)
preview_segmentation('darkness', DARKNESS_PREVIEW_INDICES, DARKNESS_SEVERITY)


#### 1.11.5.3 U-Net: Повышенная яркость: оценка


In [ ]:
BRIGHTNESS_SEVERITY = 1
if BRIGHTNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('BRIGHTNESS_SEVERITY must be 1, 2, or 3')
os.environ['BRIGHTNESS_SEVERITY'] = str(BRIGHTNESS_SEVERITY)
print('Increased brightness severity:', BRIGHTNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


#### 1.11.5.4 U-Net: Повышенная яркость: предпросмотры сегментации


In [ ]:
BRIGHTNESS_PREVIEW_INDICES = [0]
BRIGHTNESS_SEVERITY = globals().get('BRIGHTNESS_SEVERITY', 1)
preview_segmentation('brightness', BRIGHTNESS_PREVIEW_INDICES, BRIGHTNESS_SEVERITY)


#### 1.11.5.5 U-Net: Гауссово размытие: оценка


In [ ]:
GAUSSIAN_BLUR_SEVERITY = 1
if GAUSSIAN_BLUR_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_BLUR_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(GAUSSIAN_BLUR_SEVERITY)
print('Gaussian blur severity:', GAUSSIAN_BLUR_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


#### 1.11.5.6 U-Net: Гауссово размытие: предпросмотры сегментации


In [ ]:
GAUSSIAN_BLUR_PREVIEW_INDICES = [0]
GAUSSIAN_BLUR_SEVERITY = globals().get('GAUSSIAN_BLUR_SEVERITY', 1)
preview_segmentation('gaussian_blur', GAUSSIAN_BLUR_PREVIEW_INDICES, GAUSSIAN_BLUR_SEVERITY)


#### 1.11.5.7 U-Net: Гауссов шум: оценка


In [ ]:
GAUSSIAN_NOISE_SEVERITY = 1
if GAUSSIAN_NOISE_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_NOISE_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(GAUSSIAN_NOISE_SEVERITY)
print('Gaussian noise severity:', GAUSSIAN_NOISE_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


#### 1.11.5.8 U-Net: Гауссов шум: предпросмотры сегментации


In [ ]:
GAUSSIAN_NOISE_PREVIEW_INDICES = [0]
GAUSSIAN_NOISE_SEVERITY = globals().get('GAUSSIAN_NOISE_SEVERITY', 1)
preview_segmentation('gaussian_noise', GAUSSIAN_NOISE_PREVIEW_INDICES, GAUSSIAN_NOISE_SEVERITY)


#### 1.11.5.9 U-Net: JPEG-сжатие: оценка


In [ ]:
JPEG_COMPRESSION_SEVERITY = 1
if JPEG_COMPRESSION_SEVERITY not in {1, 2, 3}:
    raise ValueError('JPEG_COMPRESSION_SEVERITY must be 1, 2, or 3')
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(JPEG_COMPRESSION_SEVERITY)
print('JPEG compression severity:', JPEG_COMPRESSION_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


#### 1.11.5.10 U-Net: JPEG-сжатие: предпросмотры сегментации


In [ ]:
JPEG_COMPRESSION_PREVIEW_INDICES = [0]
JPEG_COMPRESSION_SEVERITY = globals().get('JPEG_COMPRESSION_SEVERITY', 1)
preview_segmentation('jpeg_compression', JPEG_COMPRESSION_PREVIEW_INDICES, JPEG_COMPRESSION_SEVERITY)


#### 1.11.5.11 U-Net: Пелена тумана: оценка


In [ ]:
FOG_SEVERITY = 1
if FOG_SEVERITY not in {1, 2, 3}:
    raise ValueError('FOG_SEVERITY must be 1, 2, or 3')
os.environ['FOG_SEVERITY'] = str(FOG_SEVERITY)
print('Fog veil severity:', FOG_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


#### 1.11.5.12 U-Net: Пелена тумана: предпросмотры сегментации


In [ ]:
FOG_PREVIEW_INDICES = [0]
FOG_SEVERITY = globals().get('FOG_SEVERITY', 1)
preview_segmentation('fog', FOG_PREVIEW_INDICES, FOG_SEVERITY)


### 1.11.6 U-Net: пакетная оценка искажений для всех severity

Необязательная вспомогательная ячейка: запускает все оценки искажений для этой модели. Clean-оценка остаётся в существующей clean-ячейке выше.


In [ ]:
BENCHMARK_SEVERITIES = [1, 2, 3]
if not BENCHMARK_SEVERITIES:
    raise ValueError('BENCHMARK_SEVERITIES must contain at least one severity')
for severity in BENCHMARK_SEVERITIES:
    if severity not in {1, 2, 3}:
        raise ValueError('BENCHMARK_SEVERITIES values must be 1, 2, or 3')

os.environ['BENCHMARK_SEVERITIES'] = ' '.join(str(severity) for severity in BENCHMARK_SEVERITIES)
print('Benchmark severities:', BENCHMARK_SEVERITIES)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"

for severity in $BENCHMARK_SEVERITIES; do
  for condition in darkness brightness gaussian_blur gaussian_noise jpeg_compression fog; do
    echo "Evaluation: ${condition} severity ${severity}"
    python -u scripts/evaluate_model.py \
      --config "$RUN_CONFIG" \
      --replace-existing \
      --condition "$condition" \
      --severity "$severity" \
      2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_${condition}_s${severity}.log"
  done
done


### 1.11.7 U-Net: пакетный предпросмотр выбранных индексов для всех severity

Необязательная вспомогательная ячейка: показывает в Colab clean-preview для каждого выбранного индекса, затем каждое искажение и severity. Файлы на Google Drive не создаются.


In [ ]:
BENCHMARK_PREVIEW_INDICES = [0]
BENCHMARK_PREVIEW_SEVERITIES = [1, 2, 3]
if not BENCHMARK_PREVIEW_INDICES:
    raise ValueError('BENCHMARK_PREVIEW_INDICES must contain at least one image index')
if not BENCHMARK_PREVIEW_SEVERITIES:
    raise ValueError('BENCHMARK_PREVIEW_SEVERITIES must contain at least one severity')
for severity in BENCHMARK_PREVIEW_SEVERITIES:
    if severity not in {1, 2, 3}:
        raise ValueError('BENCHMARK_PREVIEW_SEVERITIES values must be 1, 2, or 3')

print('Benchmark preview indices:', BENCHMARK_PREVIEW_INDICES)
print('Benchmark preview severities:', BENCHMARK_PREVIEW_SEVERITIES)


In [ ]:
benchmark_preview_conditions = [
    'darkness',
    'brightness',
    'gaussian_blur',
    'gaussian_noise',
    'jpeg_compression',
    'fog',
]

preview_segmentation('clean', BENCHMARK_PREVIEW_INDICES)
for severity in BENCHMARK_PREVIEW_SEVERITIES:
    for condition in benchmark_preview_conditions:
        preview_segmentation(
            condition,
            BENCHMARK_PREVIEW_INDICES,
            severity,
        )


### 1.11.8 U-Net: экспорт исходников для дипломных иллюстраций

Необязательный блок после отбора подходящих сцен. Только этот механизм сохраняет изображения: компоненты записываются в `predictions/qualitative`.


In [ ]:
QUALITATIVE_EXPORT_INDICES = [0]
QUALITATIVE_EXPORT_CONDITIONS = [
    'clean',
    'darkness',
    'brightness',
    'gaussian_blur',
    'gaussian_noise',
    'jpeg_compression',
    'fog',
]
QUALITATIVE_EXPORT_SEVERITIES = [1, 2, 3]

export_command = [
    'python',
    'scripts/export_qualitative.py',
    '--config',
    str(RUN_CONFIG),
    '--indices',
    *[str(index) for index in QUALITATIVE_EXPORT_INDICES],
    '--conditions',
    *QUALITATIVE_EXPORT_CONDITIONS,
    '--severities',
    *[str(severity) for severity in QUALITATIVE_EXPORT_SEVERITIES],
]
run_command_to_log(
    export_command,
    LOG_DIR / f'qualitative_export_{RUN_NAME}.log',
)
print('Qualitative export:', RUN_DIR / 'predictions' / 'qualitative')


## 1.12 DeepLabV3+

Сверните этот раздел, когда работаете с другой моделью.


### 1.12.1 DeepLabV3+: параметры


In [ ]:
DEEPLABV3PLUS_SETTINGS = {
    'model': {
        'name': 'deeplabv3plus',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    'run': {
        'kind': 'baseline',
        'source_baseline_run': None,
    },
    'augmentation': BASELINE_AUGMENTATION,
    'training': {
        'epochs': 8,
        'batch_size': 8,
        'num_workers': 2,
        'log_interval': 25,
        'learning_rate': 0.0003,
        'weight_decay': 0.0001,
        'device': 'auto',
        'mixed_precision': True,
    },
    'evaluation': {
        'batch_size': 8,
    },
}

prepare_model_run(
    selected_model='deeplabv3plus',
    run_name='deeplabv3plus_experiment_01',
    resume_training=False,
    continue_from_run=None,
    init_checkpoint='last',
    model_settings=DEEPLABV3PLUS_SETTINGS,
    update_corruption_settings=False,
)


### 1.12.2 DeepLabV3+: обучение, resume или continue


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -n "$CONTINUE_FROM_RUN" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --continue-from-run "$CONTINUE_FROM_RUN" --init-checkpoint "$INIT_CHECKPOINT" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
elif [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi


In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


### 1.12.3 DeepLabV3+: clean-оценка и предпросмотры


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"


In [ ]:
CLEAN_PREVIEW_INDICES = [0]
preview_segmentation('clean', CLEAN_PREVIEW_INDICES)


### 1.12.4 DeepLabV3+: сохранённые результаты


In [ ]:
import pandas as pd

print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
history_path = RUN_DIR / 'metrics' / 'training_history.csv'
display(pd.read_csv(history_path))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

for figure in create_training_curve_figures(history_path):
    display(figure)
    plt.close(figure)


### 1.12.5 DeepLabV3+: ручные оценки и предпросмотры искажений

В этих блоках вручную выбирается severity и список индексов для preview каждого искажения.


#### 1.12.5.1 DeepLabV3+: Darkness: оценка


In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY must be 1, 2, or 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


#### 1.12.5.2 DeepLabV3+: Darkness: предпросмотры сегментации


In [ ]:
DARKNESS_PREVIEW_INDICES = [0]
DARKNESS_SEVERITY = globals().get('DARKNESS_SEVERITY', 1)
preview_segmentation('darkness', DARKNESS_PREVIEW_INDICES, DARKNESS_SEVERITY)


#### 1.12.5.3 DeepLabV3+: Повышенная яркость: оценка


In [ ]:
BRIGHTNESS_SEVERITY = 1
if BRIGHTNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('BRIGHTNESS_SEVERITY must be 1, 2, or 3')
os.environ['BRIGHTNESS_SEVERITY'] = str(BRIGHTNESS_SEVERITY)
print('Increased brightness severity:', BRIGHTNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


#### 1.12.5.4 DeepLabV3+: Повышенная яркость: предпросмотры сегментации


In [ ]:
BRIGHTNESS_PREVIEW_INDICES = [0]
BRIGHTNESS_SEVERITY = globals().get('BRIGHTNESS_SEVERITY', 1)
preview_segmentation('brightness', BRIGHTNESS_PREVIEW_INDICES, BRIGHTNESS_SEVERITY)


#### 1.12.5.5 DeepLabV3+: Гауссово размытие: оценка


In [ ]:
GAUSSIAN_BLUR_SEVERITY = 1
if GAUSSIAN_BLUR_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_BLUR_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(GAUSSIAN_BLUR_SEVERITY)
print('Gaussian blur severity:', GAUSSIAN_BLUR_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


#### 1.12.5.6 DeepLabV3+: Гауссово размытие: предпросмотры сегментации


In [ ]:
GAUSSIAN_BLUR_PREVIEW_INDICES = [0]
GAUSSIAN_BLUR_SEVERITY = globals().get('GAUSSIAN_BLUR_SEVERITY', 1)
preview_segmentation('gaussian_blur', GAUSSIAN_BLUR_PREVIEW_INDICES, GAUSSIAN_BLUR_SEVERITY)


#### 1.12.5.7 DeepLabV3+: Гауссов шум: оценка


In [ ]:
GAUSSIAN_NOISE_SEVERITY = 1
if GAUSSIAN_NOISE_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_NOISE_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(GAUSSIAN_NOISE_SEVERITY)
print('Gaussian noise severity:', GAUSSIAN_NOISE_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


#### 1.12.5.8 DeepLabV3+: Гауссов шум: предпросмотры сегментации


In [ ]:
GAUSSIAN_NOISE_PREVIEW_INDICES = [0]
GAUSSIAN_NOISE_SEVERITY = globals().get('GAUSSIAN_NOISE_SEVERITY', 1)
preview_segmentation('gaussian_noise', GAUSSIAN_NOISE_PREVIEW_INDICES, GAUSSIAN_NOISE_SEVERITY)


#### 1.12.5.9 DeepLabV3+: JPEG-сжатие: оценка


In [ ]:
JPEG_COMPRESSION_SEVERITY = 1
if JPEG_COMPRESSION_SEVERITY not in {1, 2, 3}:
    raise ValueError('JPEG_COMPRESSION_SEVERITY must be 1, 2, or 3')
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(JPEG_COMPRESSION_SEVERITY)
print('JPEG compression severity:', JPEG_COMPRESSION_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


#### 1.12.5.10 DeepLabV3+: JPEG-сжатие: предпросмотры сегментации


In [ ]:
JPEG_COMPRESSION_PREVIEW_INDICES = [0]
JPEG_COMPRESSION_SEVERITY = globals().get('JPEG_COMPRESSION_SEVERITY', 1)
preview_segmentation('jpeg_compression', JPEG_COMPRESSION_PREVIEW_INDICES, JPEG_COMPRESSION_SEVERITY)


#### 1.12.5.11 DeepLabV3+: Пелена тумана: оценка


In [ ]:
FOG_SEVERITY = 1
if FOG_SEVERITY not in {1, 2, 3}:
    raise ValueError('FOG_SEVERITY must be 1, 2, or 3')
os.environ['FOG_SEVERITY'] = str(FOG_SEVERITY)
print('Fog veil severity:', FOG_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


#### 1.12.5.12 DeepLabV3+: Пелена тумана: предпросмотры сегментации


In [ ]:
FOG_PREVIEW_INDICES = [0]
FOG_SEVERITY = globals().get('FOG_SEVERITY', 1)
preview_segmentation('fog', FOG_PREVIEW_INDICES, FOG_SEVERITY)


### 1.12.6 DeepLabV3+: пакетная оценка искажений для всех severity

Необязательная вспомогательная ячейка: запускает все оценки искажений для этой модели. Clean-оценка остаётся в существующей clean-ячейке выше.


In [ ]:
BENCHMARK_SEVERITIES = [1, 2, 3]
if not BENCHMARK_SEVERITIES:
    raise ValueError('BENCHMARK_SEVERITIES must contain at least one severity')
for severity in BENCHMARK_SEVERITIES:
    if severity not in {1, 2, 3}:
        raise ValueError('BENCHMARK_SEVERITIES values must be 1, 2, or 3')

os.environ['BENCHMARK_SEVERITIES'] = ' '.join(str(severity) for severity in BENCHMARK_SEVERITIES)
print('Benchmark severities:', BENCHMARK_SEVERITIES)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"

for severity in $BENCHMARK_SEVERITIES; do
  for condition in darkness brightness gaussian_blur gaussian_noise jpeg_compression fog; do
    echo "Evaluation: ${condition} severity ${severity}"
    python -u scripts/evaluate_model.py \
      --config "$RUN_CONFIG" \
      --replace-existing \
      --condition "$condition" \
      --severity "$severity" \
      2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_${condition}_s${severity}.log"
  done
done


### 1.12.7 DeepLabV3+: пакетный предпросмотр выбранных индексов для всех severity

Необязательная вспомогательная ячейка: показывает в Colab clean-preview для каждого выбранного индекса, затем каждое искажение и severity. Файлы на Google Drive не создаются.


In [ ]:
BENCHMARK_PREVIEW_INDICES = [0]
BENCHMARK_PREVIEW_SEVERITIES = [1, 2, 3]
if not BENCHMARK_PREVIEW_INDICES:
    raise ValueError('BENCHMARK_PREVIEW_INDICES must contain at least one image index')
if not BENCHMARK_PREVIEW_SEVERITIES:
    raise ValueError('BENCHMARK_PREVIEW_SEVERITIES must contain at least one severity')
for severity in BENCHMARK_PREVIEW_SEVERITIES:
    if severity not in {1, 2, 3}:
        raise ValueError('BENCHMARK_PREVIEW_SEVERITIES values must be 1, 2, or 3')

print('Benchmark preview indices:', BENCHMARK_PREVIEW_INDICES)
print('Benchmark preview severities:', BENCHMARK_PREVIEW_SEVERITIES)


In [ ]:
benchmark_preview_conditions = [
    'darkness',
    'brightness',
    'gaussian_blur',
    'gaussian_noise',
    'jpeg_compression',
    'fog',
]

preview_segmentation('clean', BENCHMARK_PREVIEW_INDICES)
for severity in BENCHMARK_PREVIEW_SEVERITIES:
    for condition in benchmark_preview_conditions:
        preview_segmentation(
            condition,
            BENCHMARK_PREVIEW_INDICES,
            severity,
        )


### 1.12.8 DeepLabV3+: экспорт исходников для дипломных иллюстраций

Необязательный блок после отбора подходящих сцен. Только этот механизм сохраняет изображения: компоненты записываются в `predictions/qualitative`.


In [ ]:
QUALITATIVE_EXPORT_INDICES = [0]
QUALITATIVE_EXPORT_CONDITIONS = [
    'clean',
    'darkness',
    'brightness',
    'gaussian_blur',
    'gaussian_noise',
    'jpeg_compression',
    'fog',
]
QUALITATIVE_EXPORT_SEVERITIES = [1, 2, 3]

export_command = [
    'python',
    'scripts/export_qualitative.py',
    '--config',
    str(RUN_CONFIG),
    '--indices',
    *[str(index) for index in QUALITATIVE_EXPORT_INDICES],
    '--conditions',
    *QUALITATIVE_EXPORT_CONDITIONS,
    '--severities',
    *[str(severity) for severity in QUALITATIVE_EXPORT_SEVERITIES],
]
run_command_to_log(
    export_command,
    LOG_DIR / f'qualitative_export_{RUN_NAME}.log',
)
print('Qualitative export:', RUN_DIR / 'predictions' / 'qualitative')


## 1.13 PSPNet

Сверните этот раздел, когда работаете с другой моделью.


### 1.13.1 PSPNet: параметры


In [ ]:
PSPNET_SETTINGS = {
    'model': {
        'name': 'pspnet',
        'encoder_name': 'resnet34',
        'encoder_weights': 'imagenet',
    },
    'run': {
        'kind': 'baseline',
        'source_baseline_run': None,
    },
    'augmentation': BASELINE_AUGMENTATION,
    'training': {
        'epochs': 8,
        'batch_size': 8,
        'num_workers': 2,
        'log_interval': 25,
        'learning_rate': 0.0003,
        'weight_decay': 0.0001,
        'device': 'auto',
        'mixed_precision': True,
    },
    'evaluation': {
        'batch_size': 8,
    },
}

prepare_model_run(
    selected_model='pspnet',
    run_name='pspnet_experiment_01',
    resume_training=False,
    continue_from_run=None,
    init_checkpoint='last',
    model_settings=PSPNET_SETTINGS,
    update_corruption_settings=False,
)


### 1.13.2 PSPNet: обучение, resume или continue


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -n "$CONTINUE_FROM_RUN" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --continue-from-run "$CONTINUE_FROM_RUN" --init-checkpoint "$INIT_CHECKPOINT" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
elif [[ "$RESUME_TRAINING" == "1" ]]; then
  python -u scripts/train_model.py --config "$RUN_CONFIG" --resume 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
else
  python -u scripts/train_model.py --config "$RUN_CONFIG" 2>&1 | tee -a "$LOG_DIR/train_$RUN_NAME.log"
fi


In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


### 1.13.3 PSPNet: clean-оценка и предпросмотры


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition clean 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_clean.log"


In [ ]:
CLEAN_PREVIEW_INDICES = [0]
preview_segmentation('clean', CLEAN_PREVIEW_INDICES)


### 1.13.4 PSPNet: сохранённые результаты


In [ ]:
import pandas as pd

print('Results dir:', RUN_DIR)
print('Model dir:', MODEL_DIR)
history_path = RUN_DIR / 'metrics' / 'training_history.csv'
display(pd.read_csv(history_path))
display(pd.read_csv(RUN_DIR / 'metrics' / 'evaluation_results.csv'))

for figure in create_training_curve_figures(history_path):
    display(figure)
    plt.close(figure)


### 1.13.5 PSPNet: ручные оценки и предпросмотры искажений

В этих блоках вручную выбирается severity и список индексов для preview каждого искажения.


#### 1.13.5.1 PSPNet: Darkness: оценка


In [ ]:
DARKNESS_SEVERITY = 1
if DARKNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('DARKNESS_SEVERITY must be 1, 2, or 3')
os.environ['DARKNESS_SEVERITY'] = str(DARKNESS_SEVERITY)
print('Darkness severity:', DARKNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition darkness --severity "$DARKNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_darkness_s${DARKNESS_SEVERITY}.log"


#### 1.13.5.2 PSPNet: Darkness: предпросмотры сегментации


In [ ]:
DARKNESS_PREVIEW_INDICES = [0]
DARKNESS_SEVERITY = globals().get('DARKNESS_SEVERITY', 1)
preview_segmentation('darkness', DARKNESS_PREVIEW_INDICES, DARKNESS_SEVERITY)


#### 1.13.5.3 PSPNet: Повышенная яркость: оценка


In [ ]:
BRIGHTNESS_SEVERITY = 1
if BRIGHTNESS_SEVERITY not in {1, 2, 3}:
    raise ValueError('BRIGHTNESS_SEVERITY must be 1, 2, or 3')
os.environ['BRIGHTNESS_SEVERITY'] = str(BRIGHTNESS_SEVERITY)
print('Increased brightness severity:', BRIGHTNESS_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition brightness --severity "$BRIGHTNESS_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_brightness_s${BRIGHTNESS_SEVERITY}.log"


#### 1.13.5.4 PSPNet: Повышенная яркость: предпросмотры сегментации


In [ ]:
BRIGHTNESS_PREVIEW_INDICES = [0]
BRIGHTNESS_SEVERITY = globals().get('BRIGHTNESS_SEVERITY', 1)
preview_segmentation('brightness', BRIGHTNESS_PREVIEW_INDICES, BRIGHTNESS_SEVERITY)


#### 1.13.5.5 PSPNet: Гауссово размытие: оценка


In [ ]:
GAUSSIAN_BLUR_SEVERITY = 1
if GAUSSIAN_BLUR_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_BLUR_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_BLUR_SEVERITY'] = str(GAUSSIAN_BLUR_SEVERITY)
print('Gaussian blur severity:', GAUSSIAN_BLUR_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_blur --severity "$GAUSSIAN_BLUR_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_blur_s${GAUSSIAN_BLUR_SEVERITY}.log"


#### 1.13.5.6 PSPNet: Гауссово размытие: предпросмотры сегментации


In [ ]:
GAUSSIAN_BLUR_PREVIEW_INDICES = [0]
GAUSSIAN_BLUR_SEVERITY = globals().get('GAUSSIAN_BLUR_SEVERITY', 1)
preview_segmentation('gaussian_blur', GAUSSIAN_BLUR_PREVIEW_INDICES, GAUSSIAN_BLUR_SEVERITY)


#### 1.13.5.7 PSPNet: Гауссов шум: оценка


In [ ]:
GAUSSIAN_NOISE_SEVERITY = 1
if GAUSSIAN_NOISE_SEVERITY not in {1, 2, 3}:
    raise ValueError('GAUSSIAN_NOISE_SEVERITY must be 1, 2, or 3')
os.environ['GAUSSIAN_NOISE_SEVERITY'] = str(GAUSSIAN_NOISE_SEVERITY)
print('Gaussian noise severity:', GAUSSIAN_NOISE_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition gaussian_noise --severity "$GAUSSIAN_NOISE_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_gaussian_noise_s${GAUSSIAN_NOISE_SEVERITY}.log"


#### 1.13.5.8 PSPNet: Гауссов шум: предпросмотры сегментации


In [ ]:
GAUSSIAN_NOISE_PREVIEW_INDICES = [0]
GAUSSIAN_NOISE_SEVERITY = globals().get('GAUSSIAN_NOISE_SEVERITY', 1)
preview_segmentation('gaussian_noise', GAUSSIAN_NOISE_PREVIEW_INDICES, GAUSSIAN_NOISE_SEVERITY)


#### 1.13.5.9 PSPNet: JPEG-сжатие: оценка


In [ ]:
JPEG_COMPRESSION_SEVERITY = 1
if JPEG_COMPRESSION_SEVERITY not in {1, 2, 3}:
    raise ValueError('JPEG_COMPRESSION_SEVERITY must be 1, 2, or 3')
os.environ['JPEG_COMPRESSION_SEVERITY'] = str(JPEG_COMPRESSION_SEVERITY)
print('JPEG compression severity:', JPEG_COMPRESSION_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition jpeg_compression --severity "$JPEG_COMPRESSION_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_jpeg_compression_s${JPEG_COMPRESSION_SEVERITY}.log"


#### 1.13.5.10 PSPNet: JPEG-сжатие: предпросмотры сегментации


In [ ]:
JPEG_COMPRESSION_PREVIEW_INDICES = [0]
JPEG_COMPRESSION_SEVERITY = globals().get('JPEG_COMPRESSION_SEVERITY', 1)
preview_segmentation('jpeg_compression', JPEG_COMPRESSION_PREVIEW_INDICES, JPEG_COMPRESSION_SEVERITY)


#### 1.13.5.11 PSPNet: Пелена тумана: оценка


In [ ]:
FOG_SEVERITY = 1
if FOG_SEVERITY not in {1, 2, 3}:
    raise ValueError('FOG_SEVERITY must be 1, 2, or 3')
os.environ['FOG_SEVERITY'] = str(FOG_SEVERITY)
print('Fog veil severity:', FOG_SEVERITY)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/evaluate_model.py --config "$RUN_CONFIG" --replace-existing --condition fog --severity "$FOG_SEVERITY" 2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_fog_s${FOG_SEVERITY}.log"


#### 1.13.5.12 PSPNet: Пелена тумана: предпросмотры сегментации


In [ ]:
FOG_PREVIEW_INDICES = [0]
FOG_SEVERITY = globals().get('FOG_SEVERITY', 1)
preview_segmentation('fog', FOG_PREVIEW_INDICES, FOG_SEVERITY)


### 1.13.6 PSPNet: пакетная оценка искажений для всех severity

Необязательная вспомогательная ячейка: запускает все оценки искажений для этой модели. Clean-оценка остаётся в существующей clean-ячейке выше.


In [ ]:
BENCHMARK_SEVERITIES = [1, 2, 3]
if not BENCHMARK_SEVERITIES:
    raise ValueError('BENCHMARK_SEVERITIES must contain at least one severity')
for severity in BENCHMARK_SEVERITIES:
    if severity not in {1, 2, 3}:
        raise ValueError('BENCHMARK_SEVERITIES values must be 1, 2, or 3')

os.environ['BENCHMARK_SEVERITIES'] = ' '.join(str(severity) for severity in BENCHMARK_SEVERITIES)
print('Benchmark severities:', BENCHMARK_SEVERITIES)


In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"

for severity in $BENCHMARK_SEVERITIES; do
  for condition in darkness brightness gaussian_blur gaussian_noise jpeg_compression fog; do
    echo "Evaluation: ${condition} severity ${severity}"
    python -u scripts/evaluate_model.py \
      --config "$RUN_CONFIG" \
      --replace-existing \
      --condition "$condition" \
      --severity "$severity" \
      2>&1 | tee -a "$LOG_DIR/evaluate_${RUN_NAME}_${condition}_s${severity}.log"
  done
done


### 1.13.7 PSPNet: пакетный предпросмотр выбранных индексов для всех severity

Необязательная вспомогательная ячейка: показывает в Colab clean-preview для каждого выбранного индекса, затем каждое искажение и severity. Файлы на Google Drive не создаются.


In [ ]:
BENCHMARK_PREVIEW_INDICES = [0]
BENCHMARK_PREVIEW_SEVERITIES = [1, 2, 3]
if not BENCHMARK_PREVIEW_INDICES:
    raise ValueError('BENCHMARK_PREVIEW_INDICES must contain at least one image index')
if not BENCHMARK_PREVIEW_SEVERITIES:
    raise ValueError('BENCHMARK_PREVIEW_SEVERITIES must contain at least one severity')
for severity in BENCHMARK_PREVIEW_SEVERITIES:
    if severity not in {1, 2, 3}:
        raise ValueError('BENCHMARK_PREVIEW_SEVERITIES values must be 1, 2, or 3')

print('Benchmark preview indices:', BENCHMARK_PREVIEW_INDICES)
print('Benchmark preview severities:', BENCHMARK_PREVIEW_SEVERITIES)


In [ ]:
benchmark_preview_conditions = [
    'darkness',
    'brightness',
    'gaussian_blur',
    'gaussian_noise',
    'jpeg_compression',
    'fog',
]

preview_segmentation('clean', BENCHMARK_PREVIEW_INDICES)
for severity in BENCHMARK_PREVIEW_SEVERITIES:
    for condition in benchmark_preview_conditions:
        preview_segmentation(
            condition,
            BENCHMARK_PREVIEW_INDICES,
            severity,
        )


### 1.13.8 PSPNet: экспорт исходников для дипломных иллюстраций

Необязательный блок после отбора подходящих сцен. Только этот механизм сохраняет изображения: компоненты записываются в `predictions/qualitative`.


In [ ]:
QUALITATIVE_EXPORT_INDICES = [0]
QUALITATIVE_EXPORT_CONDITIONS = [
    'clean',
    'darkness',
    'brightness',
    'gaussian_blur',
    'gaussian_noise',
    'jpeg_compression',
    'fog',
]
QUALITATIVE_EXPORT_SEVERITIES = [1, 2, 3]

export_command = [
    'python',
    'scripts/export_qualitative.py',
    '--config',
    str(RUN_CONFIG),
    '--indices',
    *[str(index) for index in QUALITATIVE_EXPORT_INDICES],
    '--conditions',
    *QUALITATIVE_EXPORT_CONDITIONS,
    '--severities',
    *[str(severity) for severity in QUALITATIVE_EXPORT_SEVERITIES],
]
run_command_to_log(
    export_command,
    LOG_DIR / f'qualitative_export_{RUN_NAME}.log',
)
print('Qualitative export:', RUN_DIR / 'predictions' / 'qualitative')


## 1.14 Resume и continue обучения

Resume после обрыва runtime:

1. Снова выполните разделы 1.1–1.8.
2. В секции параметров модели укажите старый `RUN_NAME`, `RESUME_TRAINING = True`, `CONTINUE_FROM_RUN = None`.
3. Запустите ячейку обучения этой модели. Будут повторно использованы существующие `run_config.yaml`, `last.pt`, `training_history.csv`, optimizer и scaler.

Continue уже завершённой модели в новый run:

1. В секции параметров модели задайте новый `RUN_NAME`.
2. Укажите `RESUME_TRAINING = False`.
3. Укажите `CONTINUE_FROM_RUN` как имя старого run или абсолютный путь к старой папке run.
4. Выберите `INIT_CHECKPOINT = "last"` или `"best"`.
5. В `MODEL_SETTINGS` укажите в `training.epochs` число дополнительных эпох. Новый run сохранит общий target epochs в `run_config.yaml`.

Clean-оценку и любую выбранную оценку искажения можно запускать повторно: каждая оценка сохраняется отдельно.
